In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

'''url = input("Enter webpage URL: ")

html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

images = soup.find_all("img")

for img in images:
    image_url = img.get("src")

    if image_url:
        image_url = urljoin(url, image_url)

        print(image_url)
        display(Image(url=image_url, width=300))

'''

'url = input("Enter webpage URL: ")\n\nhtml = requests.get(url).text\nsoup = BeautifulSoup(html, "html.parser")\n\nimages = soup.find_all("img")\n\nfor img in images:\n    image_url = img.get("src")\n\n    if image_url:\n        image_url = urljoin(url, image_url)\n\n        print(image_url)\n        display(Image(url=image_url, width=300))\n\n'

In [10]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from PIL import Image, ImageTk
from io import BytesIO
import os


In [11]:
def extract_images():
    global images
    images = []

    url = url_entry.get()

    try:
        soup = BeautifulSoup(requests.get(url, timeout=15).text, "html.parser")

        for tag in soup.find_all("img"):
            src = tag.get("src") or tag.get("data-src")

            if not src:
                continue

            try:
                link = urljoin(url, src)
                data = requests.get(link, timeout=10)
                data.raise_for_status()

                img = Image.open(BytesIO(data.content))
                images.append({
                    "url": link,
                    "data": data.content,
                    "width": img.width,
                    "height": img.height,
                    "type": img.format.upper(),
                    "size": len(data.content) / 1024
                })
            except Exception:
                continue

        show_images(images)

    except Exception as e:
        messagebox.showerror("Error", str(e))



In [12]:
def filter_images():
    try:
        width = int(width_entry.get() or 0)
        height = int(height_entry.get() or 0)
        file_type = type_box.get()
        sort_by = sort_box.get()

        result = [
            img for img in images
            if img["width"] >= width
            and img["height"] >= height
            and (file_type == "All" or img["type"] == file_type)
        ]

        if sort_by == "Width":
            result.sort(key=lambda x: x["width"], reverse=True)
        elif sort_by == "Height":
            result.sort(key=lambda x: x["height"], reverse=True)
        else:
            result.sort(
                key=lambda x: x["width"] * x["height"],
                reverse=True
            )

        show_images(result)

    except ValueError:
        messagebox.showerror("Error", "Enter valid resolution values.")



In [13]:
def show_images(data):
    for widget in gallery.winfo_children():
        widget.destroy()

    for i, img in enumerate(data):
        box = tk.Frame(gallery, bd=1, relief="solid", padx=5, pady=5)
        box.grid(row=i // 3, column=i % 3, padx=5, pady=5)

        picture = Image.open(BytesIO(img["data"]))
        picture.thumbnail((140, 100))
        photo = ImageTk.PhotoImage(picture)

        label = tk.Label(box, image=photo)
        label.image = photo
        label.pack()

        tk.Label(
            box,
            text=f'{img["width"]} x {img["height"]}\n'
                 f'{img["type"]} | {img["size"]:.1f} KB'
        ).pack()

        var = tk.BooleanVar()
        tk.Checkbutton(box, text="Select", variable=var).pack()

        img["var"] = var

    status.config(text=f"Images displayed: {len(data)}")


In [14]:
def download_images():
    selected = [img for img in images if img.get("var") and img["var"].get()]

    if not selected:
        messagebox.showwarning("Warning", "Select some images first.")
        return

    folder = filedialog.askdirectory()

    if folder:
        for i, img in enumerate(selected, 1):
            ext = img["type"].lower()
            if ext == "jpeg":
                ext = "jpg"

            with open(os.path.join(folder, f"image_{i}.{ext}"), "wb") as f:
                f.write(img["data"])

        messagebox.showinfo("Success", "Images downloaded!")



In [15]:
root = tk.Tk()
root.title("Web Image Extractor")
root.geometry("750x650")

tk.Label(root, text="Web Image Extractor", font=("Arial", 18, "bold")).pack(pady=10)

url_entry = tk.Entry(root, width=65)
url_entry.pack(pady=5)
url_entry.insert(0, "https://")

tk.Button(root, text="Extract Images", command=extract_images).pack(pady=5)

# Filters
filters = tk.Frame(root)
filters.pack(pady=10)

tk.Label(filters, text="Min Width:").grid(row=0, column=0)
width_entry = tk.Entry(filters, width=8)
width_entry.insert(0, "0")
width_entry.grid(row=0, column=1, padx=5)

tk.Label(filters, text="Min Height:").grid(row=0, column=2)
height_entry = tk.Entry(filters, width=8)
height_entry.insert(0, "0")
height_entry.grid(row=0, column=3, padx=5)

tk.Label(filters, text="File Type:").grid(row=1, column=0, pady=5)
type_box = ttk.Combobox(
    filters, values=["All", "JPEG", "PNG", "GIF", "WEBP"],
    state="readonly", width=10
)
type_box.set("All")
type_box.grid(row=1, column=1)

tk.Label(filters, text="Sort By:").grid(row=1, column=2)
sort_box = ttk.Combobox(
    filters, values=["Resolution", "Width", "Height"],
    state="readonly", width=10
)
sort_box.set("Resolution")
sort_box.grid(row=1, column=3)

tk.Button(root, text="Apply Filters", command=filter_images).pack(pady=5)

# Image gallery
canvas = tk.Canvas(root)
scrollbar = ttk.Scrollbar(root, orient="vertical", command=canvas.yview)
gallery = tk.Frame(canvas)

gallery.bind(
    "<Configure>",
    lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
)

canvas.create_window((0, 0), window=gallery, anchor="nw")
canvas.configure(yscrollcommand=scrollbar.set)

canvas.pack(side="left", fill="both", expand=True, padx=10, pady=5)
scrollbar.pack(side="right", fill="y")

status = tk.Label(root, text="Ready")
status.pack(side="bottom", pady=5)

tk.Button(
    root, text="Download Selected Images",
    command=download_images
).pack(side="bottom", pady=5)

root.mainloop()